In [ ]:
import pandas as pd


val_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/TFG/validation.tsv", sep="\t")
test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/TFG/test.tsv", sep="\t")
train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/TFG/train.tsv", sep="\t")
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(149043, 124)
(31938, 124)
(31938, 124)


In [ ]:
import json
import numpy as np
train_df["fingerprint"] = train_df["fingerprint"].apply(
    lambda x: np.array(json.loads(x), dtype=np.float32)
)
val_df["fingerprint"] = val_df["fingerprint"].apply(
    lambda x: np.array(json.loads(x), dtype=np.float32)
)
test_df["fingerprint"] = test_df["fingerprint"].apply(
    lambda x: np.array(json.loads(x), dtype=np.float32)
)

In [ ]:
import ast
import joblib
import numpy as np
import pandas as pd
import torch

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer


# =========================================================
# 1. Función para convertir fingerprint a lista de bits
# =========================================================

def parse_fingerprint(fp):
    """
    Convierte la columna fingerprint a lista de números.

    Soporta:
    - listas reales de Python
    - strings del tipo "[0, 1, 0, ...]"
    - arrays / listas / tuplas
    """

    if isinstance(fp, list):
        return fp

    if isinstance(fp, tuple):
        return list(fp)

    if isinstance(fp, np.ndarray):
        return fp.tolist()

    if isinstance(fp, str):
        fp = fp.strip()

        try:
            parsed = ast.literal_eval(fp)

            if isinstance(parsed, (list, tuple, np.ndarray)):
                return list(parsed)

        except Exception:
            pass

    raise ValueError(f"No se pudo parsear fingerprint: {fp}")


# =========================================================
# 2. Función para expandir fingerprint
# =========================================================

def expand_fingerprint_column(df, fingerprint_col="fingerprint"):
    """
    Expande la columna fingerprint a columnas:
    fp_0, fp_1, ..., fp_n
    """

    fp_lists = df[fingerprint_col].apply(parse_fingerprint)

    lengths = fp_lists.apply(len)

    if lengths.nunique() != 1:
        raise ValueError(
            f"Los fingerprints no tienen todos la misma longitud. "
            f"Longitudes encontradas: {sorted(lengths.unique())}"
        )

    fp_len = lengths.iloc[0]

    fp_matrix = np.vstack(fp_lists.values).astype(np.float32)

    fp_df = pd.DataFrame(
        fp_matrix,
        columns=[f"fp_{i}" for i in range(fp_len)],
        index=df.index
    )

    return fp_df


# =========================================================
# 3. Función auxiliar para convertir a tensores PyTorch
# =========================================================

def to_torch_tensor(x, dtype=torch.float32, device=None):
    """
    Convierte numpy array / DataFrame / Series a tensor PyTorch.
    """

    if isinstance(x, pd.DataFrame) or isinstance(x, pd.Series):
        x = x.to_numpy()

    x = np.asarray(x)

    tensor = torch.tensor(x, dtype=dtype)

    if device is not None:
        tensor = tensor.to(device)

    return tensor


# =========================================================
# 4. Función para invertir escalado del target
# =========================================================

def inverse_transform_target(y_scaled, y_scaler):
    """
    Invierte la transformación del target.

    Acepta:
    - numpy array
    - pandas Series/DataFrame
    - tensor de PyTorch

    Devuelve:
    - numpy array 1D en escala original
    """

    if isinstance(y_scaled, torch.Tensor):
        y_scaled = y_scaled.detach().cpu().numpy()

    if isinstance(y_scaled, pd.Series):
        y_scaled = y_scaled.to_numpy()

    if isinstance(y_scaled, pd.DataFrame):
        y_scaled = y_scaled.to_numpy()

    y_scaled = np.asarray(y_scaled, dtype=np.float32).reshape(-1, 1)

    y_original = y_scaler.inverse_transform(y_scaled).reshape(-1)

    return y_original


# =========================================================
# 5. Función para separar X e y y expandir fingerprints
# =========================================================

def build_X_y(
    df,
    target_col="rt",
    id_cols=("id", "molecule_id"),
    fingerprint_col="fingerprint"
):
    """
    A partir de un dataframe, separa X e y:

    - convierte target a numérico
    - elimina filas sin target válido
    - expande fingerprint
    - elimina columnas de id, target y fingerprint original
    - concatena variables normales + fingerprint expandido
    """

    df = df.copy()

    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df = df.dropna(subset=[target_col]).copy()

    # Expandir fingerprint
    fp_df = expand_fingerprint_column(df, fingerprint_col=fingerprint_col)

    # Columnas que no deben entrar como input
    drop_cols = [target_col, fingerprint_col]

    for col in id_cols:
        if col in df.columns:
            drop_cols.append(col)

    X_base = df.drop(columns=drop_cols, errors="ignore")
    y = df[target_col].copy()

    # Añadir fingerprints expandidos
    X = pd.concat([X_base, fp_df], axis=1)

    return X, y


# =========================================================
# 6. Función principal de preprocesamiento
# =========================================================

def preprocess_rt_splits(
    train_df,
    val_df,
    test_df,
    target_col="rt",
    id_cols=("id", "molecule_id"),
    categorical_cols=("column.name", "column.usp.code"),
    fingerprint_col="fingerprint",
    scale_target=True,
    device=None
):
    """
    Preprocesa train, validation y test ya separados.

    Importante:
    - El preprocessor se ajusta SOLO con train.
    - Validation y test solo se transforman.
    - El y_scaler también se ajusta SOLO con y_train.
    """

    # =====================================================
    # A. Construir X e y para cada subset
    # =====================================================

    X_train, y_train = build_X_y(
        train_df,
        target_col=target_col,
        id_cols=id_cols,
        fingerprint_col=fingerprint_col
    )

    X_val, y_val = build_X_y(
        val_df,
        target_col=target_col,
        id_cols=id_cols,
        fingerprint_col=fingerprint_col
    )

    X_test, y_test = build_X_y(
        test_df,
        target_col=target_col,
        id_cols=id_cols,
        fingerprint_col=fingerprint_col
    )

    # =====================================================
    # B. Definir columnas categóricas, numéricas y fingerprint
    # =====================================================

    categorical_cols = [c for c in categorical_cols if c in X_train.columns]

    fp_cols = [c for c in X_train.columns if c.startswith("fp_")]

    numeric_cols = [
        c for c in X_train.columns
        if c not in categorical_cols and c not in fp_cols
    ]

    # =====================================================
    # C. Pipelines de preprocesado
    # =====================================================

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    fingerprint_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", categorical_pipeline, categorical_cols),
            ("num", numeric_pipeline, numeric_cols),
            ("fp", fingerprint_pipeline, fp_cols),
        ],
        remainder="drop"
    )

    # =====================================================
    # D. Ajustar SOLO con train
    # =====================================================

    X_train_processed = preprocessor.fit_transform(X_train)

    X_val_processed = preprocessor.transform(X_val)
    X_test_processed = preprocessor.transform(X_test)

    # =====================================================
    # E. Reconstruir DataFrames procesados
    # =====================================================

    feature_names = preprocessor.get_feature_names_out()

    X_train_processed = pd.DataFrame(
        X_train_processed,
        columns=feature_names,
        index=X_train.index
    ).astype(np.float32)

    X_val_processed = pd.DataFrame(
        X_val_processed,
        columns=feature_names,
        index=X_val.index
    ).astype(np.float32)

    X_test_processed = pd.DataFrame(
        X_test_processed,
        columns=feature_names,
        index=X_test.index
    ).astype(np.float32)

    # =====================================================
    # F. Guardar y original antes de escalar
    # =====================================================

    y_train_original = y_train.astype(np.float32).copy()
    y_val_original = y_val.astype(np.float32).copy()
    y_test_original = y_test.astype(np.float32).copy()

    # =====================================================
    # G. Escalado opcional del target
    # =====================================================

    y_scaler = None

    if scale_target:
        y_scaler = StandardScaler()

        y_train_scaled = y_scaler.fit_transform(
            y_train.to_numpy().reshape(-1, 1)
        ).reshape(-1).astype(np.float32)

        y_val_scaled = y_scaler.transform(
            y_val.to_numpy().reshape(-1, 1)
        ).reshape(-1).astype(np.float32)

        y_test_scaled = y_scaler.transform(
            y_test.to_numpy().reshape(-1, 1)
        ).reshape(-1).astype(np.float32)

        y_train_processed = pd.Series(
            y_train_scaled,
            index=y_train.index,
            name=target_col
        )

        y_val_processed = pd.Series(
            y_val_scaled,
            index=y_val.index,
            name=target_col
        )

        y_test_processed = pd.Series(
            y_test_scaled,
            index=y_test.index,
            name=target_col
        )

    else:
        y_train_processed = y_train.astype(np.float32)
        y_val_processed = y_val.astype(np.float32)
        y_test_processed = y_test.astype(np.float32)

    # =====================================================
    # H. Conversión a tensores PyTorch
    # =====================================================

    X_train_tensor = to_torch_tensor(
        X_train_processed,
        dtype=torch.float32,
        device=device
    )

    X_val_tensor = to_torch_tensor(
        X_val_processed,
        dtype=torch.float32,
        device=device
    )

    X_test_tensor = to_torch_tensor(
        X_test_processed,
        dtype=torch.float32,
        device=device
    )

    y_train_tensor = to_torch_tensor(
        y_train_processed.to_numpy().reshape(-1, 1),
        dtype=torch.float32,
        device=device
    )

    y_val_tensor = to_torch_tensor(
        y_val_processed.to_numpy().reshape(-1, 1),
        dtype=torch.float32,
        device=device
    )

    y_test_tensor = to_torch_tensor(
        y_test_processed.to_numpy().reshape(-1, 1),
        dtype=torch.float32,
        device=device
    )

    return {
        "X_train_raw": X_train,
        "X_val_raw": X_val,
        "X_test_raw": X_test,

        "y_train_raw": y_train_original,
        "y_val_raw": y_val_original,
        "y_test_raw": y_test_original,

        "X_train_processed": X_train_processed,
        "X_val_processed": X_val_processed,
        "X_test_processed": X_test_processed,

        "y_train_processed": y_train_processed,
        "y_val_processed": y_val_processed,
        "y_test_processed": y_test_processed,

        "X_train_tensor": X_train_tensor,
        "X_val_tensor": X_val_tensor,
        "X_test_tensor": X_test_tensor,

        "y_train_tensor": y_train_tensor,
        "y_val_tensor": y_val_tensor,
        "y_test_tensor": y_test_tensor,

        "preprocessor": preprocessor,
        "y_scaler": y_scaler,

        "categorical_cols": categorical_cols,
        "numeric_cols": numeric_cols,
        "fingerprint_cols": fp_cols,
        "feature_names": feature_names,
    }

In [ ]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = preprocess_rt_splits(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    target_col="rt",
    id_cols=("id", "molecule_id"),
    categorical_cols=("column.name", "column.usp.code"),
    fingerprint_col="fingerprint",
    scale_target=True,
    device=device
)

print("Preprocesamiento completado.")
print("X_train:", data["X_train_tensor"].shape)
print("X_val:", data["X_val_tensor"].shape)
print("X_test:", data["X_test_tensor"].shape)

print("y_train:", data["y_train_tensor"].shape)
print("y_val:", data["y_val_tensor"].shape)
print("y_test:", data["y_test_tensor"].shape)

Preprocesamiento completado.
X_train: torch.Size([149043, 2233])
X_val: torch.Size([31938, 2233])
X_test: torch.Size([31938, 2233])
y_train: torch.Size([149043, 1])
y_val: torch.Size([31938, 1])
y_test: torch.Size([31938, 1])


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.4 MB/s eta 0:00:00


In [ ]:


import os
import glob
import copy
import shutil
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import optuna

from optuna.trial import TrialState
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



from google.colab import drive
drive.mount("/content/drive")

output_dir = "/content/drive/MyDrive/TFG_Optuna/Individual_Optuna"
os.makedirs(output_dir, exist_ok=True)

print("Results will be saved in:")
print(output_dir)


# If an old local Optuna DB exists, copy it to Drive once
local_db_files = glob.glob("optuna_individual_studies.db*")

for local_path in local_db_files:
    drive_path = os.path.join(output_dir, os.path.basename(local_path))

    if not os.path.exists(drive_path):
        shutil.copy2(local_path, drive_path)
        print(f"Copied local DB file to Drive: {drive_path}")
    else:
        print(f"Drive DB file already exists, not overwritten: {drive_path}")


#  GENERAL UTILITIES


def ensure_directory(path):
    os.makedirs(path, exist_ok=True)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def to_torch_tensor(x, dtype=torch.float32, device=None):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        x = x.to_numpy()

    x = np.asarray(x)
    tensor = torch.tensor(x, dtype=dtype)

    if device is not None:
        tensor = tensor.to(device)

    return tensor


def inverse_transform_target(y_scaled, y_scaler):
    if isinstance(y_scaled, torch.Tensor):
        y_scaled = y_scaled.detach().cpu().numpy()

    if isinstance(y_scaled, pd.Series):
        y_scaled = y_scaled.to_numpy()

    if isinstance(y_scaled, pd.DataFrame):
        y_scaled = y_scaled.to_numpy()

    y_scaled = np.asarray(y_scaled, dtype=np.float32).reshape(-1, 1)

    if y_scaler is None:
        return y_scaled.reshape(-1)

    return y_scaler.inverse_transform(y_scaled).reshape(-1)


def mean_relative_error(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs(y_true - y_pred) / denom))


def vector_to_comma_string(arr, decimals=8):
    arr = np.asarray(arr, dtype=np.float64).reshape(-1)
    return ",".join(f"{x:.{decimals}f}" for x in arr)


def curve_to_comma_string(values, decimals=8):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    return ",".join(f"{x:.{decimals}f}" for x in values)


def append_row_to_tsv(row, output_tsv_path):
    row_df = pd.DataFrame([row])
    write_header = not os.path.exists(output_tsv_path)

    row_df.to_csv(
        output_tsv_path,
        sep="\t",
        index=False,
        mode="a",
        header=write_header
    )

    try:
        os.sync()
    except Exception:
        pass


def load_finished_ids_from_results_tsv(results_tsv_path, id_col="id"):
    if not os.path.exists(results_tsv_path):
        return set()

    df = pd.read_csv(results_tsv_path, sep="\t")

    if id_col not in df.columns:
        return set()

    return set(df[id_col].dropna().astype(str).tolist())


def get_study_trial_counts(study):
    n_complete = sum(t.state == TrialState.COMPLETE for t in study.trials)
    n_pruned = sum(t.state == TrialState.PRUNED for t in study.trials)
    n_failed = sum(t.state == TrialState.FAIL for t in study.trials)
    n_running = sum(t.state == TrialState.RUNNING for t in study.trials)

    n_finished = n_complete + n_pruned + n_failed

    return {
        "n_trials_total": len(study.trials),
        "n_trials_complete": n_complete,
        "n_trials_pruned": n_pruned,
        "n_trials_failed": n_failed,
        "n_trials_running": n_running,
        "n_trials_finished": n_finished,
    }


# VALID ID SELECTION


def get_valid_ids_for_individual_optimization(
    train_df,
    val_df,
    test_df,
    id_col="id",
    min_train=30,
    min_val=30,
    min_test=30
):
    train_counts = train_df[id_col].value_counts()
    val_counts = val_df[id_col].value_counts()
    test_counts = test_df[id_col].value_counts()

    candidate_ids = sorted(
        set(train_counts.index) &
        set(val_counts.index) &
        set(test_counts.index)
    )

    valid_ids = []
    invalid_rows = []

    for selected_id in candidate_ids:
        n_train = int(train_counts.get(selected_id, 0))
        n_val = int(val_counts.get(selected_id, 0))
        n_test = int(test_counts.get(selected_id, 0))

        if n_train >= min_train and n_val >= min_val and n_test >= min_test:
            valid_ids.append(selected_id)
        else:
            invalid_rows.append({
                "id": selected_id,
                "n_train": n_train,
                "n_val": n_val,
                "n_test": n_test,
                "error": f"Insufficient rows: train={n_train}, val={n_val}, test={n_test}"
            })

    invalid_ids_df = pd.DataFrame(invalid_rows)

    return valid_ids, invalid_ids_df



#  MODEL


class RetentionTimeNNOptuna(nn.Module):
    def __init__(self, input_dim: int, hidden_dim_1: int, hidden_dim_2: int, dropout: float):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_2, 1)
        )

    def forward(self, x):
        return self.network(x)



#  DATA EXTRACTION PER ID


def extract_fingerprint_only_split_for_id(
    prepared_data,
    train_df,
    val_df,
    test_df,
    selected_id,
    id_col="id"
):
    train_idx = train_df.loc[train_df[id_col] == selected_id].index
    val_idx = val_df.loc[val_df[id_col] == selected_id].index
    test_idx = test_df.loc[test_df[id_col] == selected_id].index

    X_train_df = prepared_data["X_train_processed"].loc[train_idx]
    X_val_df = prepared_data["X_val_processed"].loc[val_idx]
    X_test_df = prepared_data["X_test_processed"].loc[test_idx]

    y_train_processed = prepared_data["y_train_processed"].loc[train_idx]
    y_val_processed = prepared_data["y_val_processed"].loc[val_idx]
    y_test_processed = prepared_data["y_test_processed"].loc[test_idx]

    y_train_raw = prepared_data["y_train_raw"].loc[train_idx]
    y_val_raw = prepared_data["y_val_raw"].loc[val_idx]
    y_test_raw = prepared_data["y_test_raw"].loc[test_idx]

    fp_feature_cols = [
        c for c in X_train_df.columns
        if c.startswith("fp__") or c.startswith("fp_")
    ]

    if len(fp_feature_cols) == 0:
        raise ValueError("No fingerprint columns found after preprocessing.")

    X_train_fp = X_train_df[fp_feature_cols].astype(np.float32)
    X_val_fp = X_val_df[fp_feature_cols].astype(np.float32)
    X_test_fp = X_test_df[fp_feature_cols].astype(np.float32)

    return {
        "X_train": X_train_fp,
        "X_val": X_val_fp,
        "X_test": X_test_fp,

        "y_train_processed": y_train_processed.astype(np.float32),
        "y_val_processed": y_val_processed.astype(np.float32),
        "y_test_processed": y_test_processed.astype(np.float32),

        "y_train_raw": np.asarray(y_train_raw, dtype=np.float32).reshape(-1),
        "y_val_raw": np.asarray(y_val_raw, dtype=np.float32).reshape(-1),
        "y_test_raw": np.asarray(y_test_raw, dtype=np.float32).reshape(-1),

        "fp_feature_cols": fp_feature_cols,
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "n_test": len(test_idx),
    }


#  ONE OPTUNA TRIAL


def train_one_trial_individual(
    trial,
    split_data,
    y_scaler,
    device,
    seed=42,
    patience=20,
    min_delta=1e-4
):
    set_seed(seed)

    learning_rate = trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.35)

    hidden_dim_pair = trial.suggest_categorical(
        "hidden_dim_pair",
        [
            "64_32",
            "64_64",
            "128_32",
            "128_64",
            "128_128",
            "256_32",
            "256_64",
            "256_128",
            "256_256",
            "512_32",
            "512_64",
            "512_128",
            "512_256",
        ]
    )

    hidden_dim_1, hidden_dim_2 = map(int, hidden_dim_pair.split("_"))

    weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    max_epochs = trial.suggest_int("max_epochs", 80, 300)

    X_train_t = to_torch_tensor(split_data["X_train"], dtype=torch.float32, device=None)
    X_val_t = to_torch_tensor(split_data["X_val"], dtype=torch.float32, device=device)

    y_train_t = to_torch_tensor(
        split_data["y_train_processed"].to_numpy().reshape(-1, 1),
        dtype=torch.float32,
        device=None
    )

    y_val_raw = split_data["y_val_raw"]

    model = RetentionTimeNNOptuna(
        input_dim=X_train_t.shape[1],
        hidden_dim_1=hidden_dim_1,
        hidden_dim_2=hidden_dim_2,
        dropout=dropout
    ).to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    train_dataset = TensorDataset(X_train_t, y_train_t)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    best_val_mae = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    train_losses = []
    val_maes = []

    for epoch in range(max_epochs):
        model.train()
        batch_losses = []

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            y_pred_batch = model(X_batch)
            train_loss = criterion(y_pred_batch, y_batch)

            train_loss.backward()
            optimizer.step()

            batch_losses.append(train_loss.item())

        mean_train_loss = float(np.mean(batch_losses))
        train_losses.append(mean_train_loss)

        model.eval()
        with torch.no_grad():
            y_val_pred_scaled = model(X_val_t)

        y_val_pred = inverse_transform_target(y_val_pred_scaled, y_scaler)
        val_mae = mean_absolute_error(y_val_raw, y_val_pred)

        val_maes.append(float(val_mae))

        trial.report(val_mae, step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_mae < best_val_mae - min_delta:
            best_val_mae = float(val_mae)
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return {
        "best_val_mae": best_val_mae,
        "best_model_state": best_model_state,
        "train_losses": train_losses,
        "val_maes": val_maes,
        "trained_epochs": len(train_losses),
        "params": {
            "learning_rate": learning_rate,
            "dropout": dropout,
            "hidden_dim_pair": hidden_dim_pair,
            "hidden_dim_1": hidden_dim_1,
            "hidden_dim_2": hidden_dim_2,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "max_epochs": max_epochs,
        }
    }


#  RETRAIN BEST MODEL AND TEST


def retrain_best_individual_model_and_test(
    best_params,
    split_data,
    y_scaler,
    device,
    seed=42,
    patience=20,
    min_delta=1e-4
):
    set_seed(seed)

    X_train_t = to_torch_tensor(split_data["X_train"], dtype=torch.float32, device=None)
    X_val_t = to_torch_tensor(split_data["X_val"], dtype=torch.float32, device=device)
    X_test_t = to_torch_tensor(split_data["X_test"], dtype=torch.float32, device=device)

    y_train_t = to_torch_tensor(
        split_data["y_train_processed"].to_numpy().reshape(-1, 1),
        dtype=torch.float32,
        device=None
    )

    y_test_t = to_torch_tensor(
        split_data["y_test_processed"].to_numpy().reshape(-1, 1),
        dtype=torch.float32,
        device=device
    )

    y_val_raw = split_data["y_val_raw"]

    hidden_dim_1, hidden_dim_2 = map(
        int,
        best_params["hidden_dim_pair"].split("_")
    )

    model = RetentionTimeNNOptuna(
        input_dim=X_train_t.shape[1],
        hidden_dim_1=hidden_dim_1,
        hidden_dim_2=hidden_dim_2,
        dropout=float(best_params["dropout"])
    ).to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(best_params["learning_rate"]),
        weight_decay=float(best_params["weight_decay"])
    )

    batch_size = int(best_params["batch_size"])
    max_epochs = int(best_params["max_epochs"])

    train_dataset = TensorDataset(X_train_t, y_train_t)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    best_val_mae = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    train_losses = []
    val_maes = []

    for epoch in range(max_epochs):
        model.train()
        batch_losses = []

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            y_pred_batch = model(X_batch)
            train_loss = criterion(y_pred_batch, y_batch)

            train_loss.backward()
            optimizer.step()

            batch_losses.append(train_loss.item())

        mean_train_loss = float(np.mean(batch_losses))
        train_losses.append(mean_train_loss)

        model.eval()
        with torch.no_grad():
            y_val_pred_scaled = model(X_val_t)

        y_val_pred = inverse_transform_target(y_val_pred_scaled, y_scaler)
        val_mae = mean_absolute_error(y_val_raw, y_val_pred)

        val_maes.append(float(val_mae))

        if val_mae < best_val_mae - min_delta:
            best_val_mae = float(val_mae)
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    model.eval()
    with torch.no_grad():
        y_test_pred_scaled = model(X_test_t)

    y_test_pred = inverse_transform_target(y_test_pred_scaled, y_scaler)
    y_test_real = inverse_transform_target(y_test_t, y_scaler)

    mae = float(mean_absolute_error(y_test_real, y_test_pred))
    rmse = float(np.sqrt(mean_squared_error(y_test_real, y_test_pred)))
    r2 = float(r2_score(y_test_real, y_test_pred))
    mre = float(mean_relative_error(y_test_real, y_test_pred))

    return {
        "model": model,
        "y_test_real": y_test_real,
        "y_test_pred": y_test_pred,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MRE": mre,
        "best_val_mae": best_val_mae,
        "last_train_loss": float(train_losses[-1]),
        "last_val_mae": float(val_maes[-1]),
        "train_losses": train_losses,
        "val_maes": val_maes,
        "trained_epochs": len(train_losses),
    }


#  SAVE TRIALS AND MODELS


def export_study_trials_to_tsv(study, selected_id, trials_dir):
    ensure_directory(trials_dir)

    trials_df = study.trials_dataframe()
    trials_df.insert(0, "id", selected_id)
    trials_df.insert(1, "study_name", study.study_name)

    trials_path = os.path.join(
        trials_dir,
        f"optuna_trials_individual_id_{selected_id}.tsv"
    )

    trials_df.to_csv(trials_path, sep="\t", index=False)

    try:
        os.sync()
    except Exception:
        pass

    return trials_path


def save_individual_model_checkpoint(
    model,
    selected_id,
    row,
    best_params,
    split_data,
    final_result,
    model_dir
):
    ensure_directory(model_dir)

    model_path = os.path.join(
        model_dir,
        f"individual_optuna_model_id_{selected_id}.pt"
    )

    checkpoint = {
        "id": selected_id,
        "model_state_dict": model.state_dict(),
        "best_params": best_params,
        "summary": row,
        "fp_feature_cols": split_data["fp_feature_cols"],
        "train_losses": final_result["train_losses"],
        "val_maes": final_result["val_maes"],
        "y_test_real": final_result["y_test_real"],
        "y_test_pred": final_result["y_test_pred"],
    }

    torch.save(checkpoint, model_path)

    try:
        os.sync()
    except Exception:
        pass

    return model_path



#  OPTIMIZE ONE ID, RESUMABLE

def optimize_one_individual_id_resumable(
    prepared_data,
    train_df,
    val_df,
    test_df,
    selected_id,
    id_col="id",
    n_trials_target=40,
    seed=42,
    patience=20,
    min_delta=1e-4,
    device=None,
    storage_path="sqlite:///optuna_individual_studies.db",
    trials_dir=None,
    model_dir=None,
    decimals=8,
    verbose=True
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    split_data = extract_fingerprint_only_split_for_id(
        prepared_data=prepared_data,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        selected_id=selected_id,
        id_col=id_col
    )

    y_scaler = prepared_data["y_scaler"]

    sampler = optuna.samplers.TPESampler(seed=seed)

    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=20,
        interval_steps=5
    )

    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        pruner=pruner,
        study_name=f"individual_id_{selected_id}",
        storage=storage_path,
        load_if_exists=True
    )

    counts_before = get_study_trial_counts(study)
    n_remaining_trials = max(
        n_trials_target - counts_before["n_trials_finished"],
        0
    )

    if verbose:
        print(f"ID {selected_id}")
        print(f"Finished trials before: {counts_before['n_trials_finished']}")
        print(f"Remaining trials to run: {n_remaining_trials}")

    def objective(trial):
        result = train_one_trial_individual(
            trial=trial,
            split_data=split_data,
            y_scaler=y_scaler,
            device=device,
            seed=seed,
            patience=patience,
            min_delta=min_delta
        )

        return result["best_val_mae"]

    if n_remaining_trials > 0:
        study.optimize(
            objective,
            n_trials=n_remaining_trials,
            show_progress_bar=False
        )

    counts_after = get_study_trial_counts(study)

    complete_trials = [
        t for t in study.trials
        if t.state == TrialState.COMPLETE
    ]

    if len(complete_trials) == 0:
        raise ValueError(f"ID {selected_id} has no completed Optuna trials.")

    best_params = study.best_trial.params

    hidden_dim_1, hidden_dim_2 = map(
        int,
        best_params["hidden_dim_pair"].split("_")
    )

    final_result = retrain_best_individual_model_and_test(
        best_params=best_params,
        split_data=split_data,
        y_scaler=y_scaler,
        device=device,
        seed=seed,
        patience=patience,
        min_delta=min_delta
    )

    row = {
        "id": selected_id,
        "n_train": split_data["n_train"],
        "n_val": split_data["n_val"],
        "n_test": split_data["n_test"],
        "n_features": int(len(split_data["fp_feature_cols"])),
        "feature_block": "fingerprint_only_optuna",

        "learning_rate": float(best_params["learning_rate"]),
        "epochs": int(final_result["trained_epochs"]),

        "MAE": final_result["MAE"],
        "RMSE": final_result["RMSE"],
        "R2": final_result["R2"],
        "MRE": final_result["MRE"],

        "best_val_loss": float(final_result["best_val_mae"]),
        "last_train_loss": float(final_result["last_train_loss"]),
        "last_val_loss": float(final_result["last_val_mae"]),

        "y_test_real_vector": vector_to_comma_string(
            final_result["y_test_real"],
            decimals=decimals
        ),
        "y_test_pred_vector": vector_to_comma_string(
            final_result["y_test_pred"],
            decimals=decimals
        ),

        "optuna_trials_target": int(n_trials_target),
        "optuna_trials_total": int(counts_after["n_trials_total"]),
        "optuna_trials_finished": int(counts_after["n_trials_finished"]),
        "optuna_trials_complete": int(counts_after["n_trials_complete"]),
        "optuna_trials_pruned": int(counts_after["n_trials_pruned"]),
        "optuna_trials_failed": int(counts_after["n_trials_failed"]),
        "optuna_trials_running": int(counts_after["n_trials_running"]),

        "best_trial_number": int(study.best_trial.number),
        "best_value_val_MAE": float(study.best_value),
        "trained_epochs_final_model": int(final_result["trained_epochs"]),

        "best_learning_rate": float(best_params["learning_rate"]),
        "best_dropout": float(best_params["dropout"]),
        "best_hidden_dim_pair": best_params["hidden_dim_pair"],
        "best_hidden_dim_1": int(hidden_dim_1),
        "best_hidden_dim_2": int(hidden_dim_2),
        "best_weight_decay": float(best_params["weight_decay"]),
        "best_batch_size": int(best_params["batch_size"]),
        "best_max_epochs": int(best_params["max_epochs"]),

        "best_val_mae": float(final_result["best_val_mae"]),
        "last_val_mae": float(final_result["last_val_mae"]),

        "train_loss_curve": curve_to_comma_string(
            final_result["train_losses"],
            decimals=decimals
        ),
        "val_mae_curve": curve_to_comma_string(
            final_result["val_maes"],
            decimals=decimals
        ),

        "study_name": study.study_name,
        "storage_path": storage_path,
    }

    if model_dir is not None:
        model_path = save_individual_model_checkpoint(
            model=final_result["model"],
            selected_id=selected_id,
            row=row,
            best_params=best_params,
            split_data=split_data,
            final_result=final_result,
            model_dir=model_dir
        )

        row["model_path"] = model_path
    else:
        row["model_path"] = None

    if trials_dir is not None:
        trials_path = export_study_trials_to_tsv(
            study=study,
            selected_id=selected_id,
            trials_dir=trials_dir
        )
    else:
        trials_path = None

    row["trials_tsv_path"] = trials_path

    if verbose:
        print(
            f"ID {selected_id} done. "
            f"Test MAE: {final_result['MAE']:.4f}. "
            f"Finished trials: {counts_after['n_trials_finished']}"
        )

    return {
        "row": row,
        "study": study,
        "model": final_result["model"],
        "split_data": split_data,
        "final_result": final_result,
    }


# RUN ALL IDS, RESUMABLE


def run_optuna_individual_for_all_valid_ids_resumable(
    prepared_data,
    train_df,
    val_df,
    test_df,
    id_col="id",
    min_train=30,
    min_val=30,
    min_test=30,
    n_trials_target=40,
    seed=42,
    patience=20,
    min_delta=1e-4,
    device=None,
    output_dir="/content/drive/MyDrive/TFG_Optuna/Individual_Optuna",
    storage_filename="optuna_individual_studies.db",
    results_tsv_filename="optuna_individual_results.tsv",
    failed_ids_tsv_filename="optuna_individual_failed_ids.tsv",
    invalid_ids_tsv_filename="optuna_individual_invalid_ids.tsv",
    trials_dirname="optuna_individual_trials",
    model_dirname="optuna_individual_models",
    decimals=8,
    save_models=True,
    verbose=True
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ensure_directory(output_dir)

    storage_db_path = os.path.join(output_dir, storage_filename)
    storage_path = f"sqlite:///{storage_db_path}"

    results_tsv_path = os.path.join(output_dir, results_tsv_filename)
    failed_ids_tsv_path = os.path.join(output_dir, failed_ids_tsv_filename)
    invalid_ids_tsv_path = os.path.join(output_dir, invalid_ids_tsv_filename)

    trials_dir = os.path.join(output_dir, trials_dirname)
    model_dir = os.path.join(output_dir, model_dirname) if save_models else None

    valid_ids, invalid_ids_df = get_valid_ids_for_individual_optimization(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        id_col=id_col,
        min_train=min_train,
        min_val=min_val,
        min_test=min_test
    )

    if len(invalid_ids_df) > 0:
        invalid_ids_df.to_csv(
            invalid_ids_tsv_path,
            sep="\t",
            index=False
        )

    finished_ids = load_finished_ids_from_results_tsv(
        results_tsv_path=results_tsv_path,
        id_col="id"
    )

    if verbose:
        print(f"Valid ids for individual Optuna: {len(valid_ids)}")
        print(f"Already finished ids in TSV: {len(finished_ids)}")
        print(f"Using device: {device}")
        print(f"Output directory: {output_dir}")
        print(f"Optuna DB: {storage_db_path}")
        print(f"Results TSV: {results_tsv_path}")

    for selected_id in valid_ids:
        selected_id_str = str(selected_id)

        if selected_id_str in finished_ids:
            if verbose:
                print(f"Skipping ID {selected_id}, already saved in results TSV")
            continue

        try:
            if verbose:
                print(f"\nOptimizing ID {selected_id}")

            result = optimize_one_individual_id_resumable(
                prepared_data=prepared_data,
                train_df=train_df,
                val_df=val_df,
                test_df=test_df,
                selected_id=selected_id,
                id_col=id_col,
                n_trials_target=n_trials_target,
                seed=seed,
                patience=patience,
                min_delta=min_delta,
                device=device,
                storage_path=storage_path,
                trials_dir=trials_dir,
                model_dir=model_dir,
                decimals=decimals,
                verbose=verbose
            )

            append_row_to_tsv(
                row=result["row"],
                output_tsv_path=results_tsv_path
            )

            finished_ids.add(selected_id_str)

            if verbose:
                print(f"ID {selected_id} saved to TSV.")

        except Exception as e:
            failed_row = {
                "id": selected_id,
                "error": str(e)
            }

            append_row_to_tsv(
                row=failed_row,
                output_tsv_path=failed_ids_tsv_path
            )

            if verbose:
                print(f"ID {selected_id} failed and was saved to failed TSV: {e}")

    if os.path.exists(results_tsv_path):
        results_df = pd.read_csv(results_tsv_path, sep="\t")
    else:
        results_df = pd.DataFrame()

    if os.path.exists(failed_ids_tsv_path):
        failed_ids_df = pd.read_csv(failed_ids_tsv_path, sep="\t")
    else:
        failed_ids_df = pd.DataFrame()

    return results_df, failed_ids_df, valid_ids


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results will be saved in:
/content/drive/MyDrive/TFG_Optuna/Individual_Optuna


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

output_dir = "/content/drive/MyDrive/TFG_Optuna/Individual_Optuna"
os.makedirs(output_dir, exist_ok=True)

print("Guardando resultados en:")
print(output_dir)
print("Existe la carpeta:", os.path.exists(output_dir))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Guardando resultados en:
/content/drive/MyDrive/TFG_Optuna/Individual_Optuna
Existe la carpeta: True


In [ ]:
# ============================================================
# 11. EXECUTION
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

results_df_optuna_ind, failed_ids_df_optuna_ind, valid_ids_optuna_ind = run_optuna_individual_for_all_valid_ids_resumable(
    prepared_data=data,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    id_col="id",
    min_train=30,
    min_val=30,
    min_test=30,
    n_trials_target=30,
    seed=42,
    patience=20,
    min_delta=1e-4,
    device=device,
    output_dir=output_dir,
    storage_filename="optuna_individual_studies.db",
    results_tsv_filename="optuna_individual_results.tsv",
    failed_ids_tsv_filename="optuna_individual_failed_ids.tsv",
    invalid_ids_tsv_filename="optuna_individual_invalid_ids.tsv",
    trials_dirname="optuna_individual_trials",
    model_dirname="optuna_individual_models",
    decimals=8,
    save_models=True,
    verbose=True
)

print("Finished.")
print("Results saved in:")
print(os.path.join(output_dir, "optuna_individual_results.tsv"))

[I 2026-06-24 20:15:14,288] Using an existing study with name 'individual_id_432' instead of creating a new one.


Device: cuda
GPU: Tesla T4
Valid ids for individual Optuna: 141
Already finished ids in TSV: 140
Using device: cuda
Output directory: /content/drive/MyDrive/TFG_Optuna/Individual_Optuna
Optuna DB: /content/drive/MyDrive/TFG_Optuna/Individual_Optuna/optuna_individual_studies.db
Results TSV: /content/drive/MyDrive/TFG_Optuna/Individual_Optuna/optuna_individual_results.tsv
Skipping ID 2, already saved in results TSV
Skipping ID 4, already saved in results TSV
Skipping ID 7, already saved in results TSV
Skipping ID 9, already saved in results TSV
Skipping ID 19, already saved in results TSV
Skipping ID 27, already saved in results TSV
Skipping ID 44, already saved in results TSV
Skipping ID 45, already saved in results TSV
Skipping ID 47, already saved in results TSV
Skipping ID 48, already saved in results TSV
Skipping ID 54, already saved in results TSV
Skipping ID 55, already saved in results TSV
Skipping ID 68, already saved in results TSV
Skipping ID 69, already saved in results TSV
S

[I 2026-06-24 20:15:18,682] Trial 14 finished with value: 3.2532923221588135 and parameters: {'learning_rate': 0.00021515598616953763, 'dropout': 0.0009039364381763781, 'hidden_dim_pair': '512_128', 'weight_decay': 6.669872006297817e-06, 'batch_size': 16, 'max_epochs': 242}. Best is trial 3 with value: 2.948085069656372.
[I 2026-06-24 20:15:31,487] Trial 15 finished with value: 2.887509346008301 and parameters: {'learning_rate': 0.00048249236841440814, 'dropout': 0.13735851223592135, 'hidden_dim_pair': '256_128', 'weight_decay': 8.987153901963636e-05, 'batch_size': 8, 'max_epochs': 203}. Best is trial 15 with value: 2.887509346008301.
[I 2026-06-24 20:15:38,725] Trial 16 finished with value: 3.022473096847534 and parameters: {'learning_rate': 0.0005153740670085196, 'dropout': 0.13933891202903173, 'hidden_dim_pair': '256_128', 'weight_decay': 7.839604206315725e-05, 'batch_size': 8, 'max_epochs': 201}. Best is trial 15 with value: 2.887509346008301.
[I 2026-06-24 20:15:45,817] Trial 17 f

ID 432 done. Test MAE: 1.8770. Finished trials: 30
ID 432 saved to TSV.
Skipping ID 433, already saved in results TSV
Skipping ID 434, already saved in results TSV
Skipping ID 435, already saved in results TSV
Skipping ID 436, already saved in results TSV
Skipping ID 437, already saved in results TSV
Skipping ID 438, already saved in results TSV
Finished.
Results saved in:
/content/drive/MyDrive/TFG_Optuna/Individual_Optuna/optuna_individual_results.tsv
